# Sistema Inteligente de Streaming de Video
## Maestría en CD & IA — UTEC | Prof. Heider Sanchez

**Corre este notebook primero.** Instala dependencias, clona el repo y verifica que todos los imports funcionen.

> ⚡ Tiempo estimado de setup: ~30 segundos

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║          CELDA 1 — SETUP GOOGLE COLAB (correr primero)      ║
# ╚══════════════════════════════════════════════════════════════╝
import os

# Detectar si estamos en Colab
IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    print('📦 Instalando dependencias...')
    os.system('pip install -q mmh3 bitarray seaborn tqdm')

    # Clonar repo solo si no existe
    if not os.path.exists('proyecto-a-e-datos'):
        print('📥 Clonando repositorio...')
        os.system('git clone https://github.com/joshilopeze-hub/proyecto-a-e-datos.git')
    else:
        print('🔄 Actualizando repositorio...')
        os.system('cd proyecto-a-e-datos && git pull')

    os.chdir('proyecto-a-e-datos')
    print(f'📂 Directorio actual: {os.getcwd()}')
else:
    # En local, ir a la raíz del proyecto
    root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    os.chdir(root)
    print(f'💻 Modo local. Directorio: {os.getcwd()}')

import sys
sys.path.insert(0, 'src')
print('✅ Setup completado!')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║          CELDA 2 — VERIFICAR IMPORTS                        ║
# ╚══════════════════════════════════════════════════════════════╝
print('Verificando imports...')

from structures.bloom_filter import BloomFilter, BotDetector
print('  ✅ BloomFilter, BotDetector')

from structures.trie import Trie
print('  ✅ Trie')

from structures.count_min_sketch import CountMinSketch, TopKTracker
print('  ✅ CountMinSketch, TopKTracker')

from structures.lsh_minhash import MinHash, LSH
print('  ✅ MinHash, LSH')

from system.lru_cache import LRUCache
print('  ✅ LRUCache')

from system.priority_queue import PriorityQueue, Event
print('  ✅ PriorityQueue, Event')

from system.stream_processor import StreamProcessor
print('  ✅ StreamProcessor')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
print('  ✅ numpy, matplotlib, seaborn, tqdm')

print('\n🎉 Todos los imports OK — puedes correr los notebooks de análisis!')

---
## Demo Rápido — Verificar que todo funciona
Prueba breve de cada estructura antes de correr los benchmarks completos.

In [ ]:
# ── BLOOM FILTER ─────────────────────────────────────────────────────────────
print('='*55)
print('1. BLOOM FILTER — Detección de usuarios duplicados/bots')
print('='*55)

bf = BloomFilter(n=10000, fp_rate=0.01)
usuarios = [f'user_{i}' for i in range(5000)]
for u in usuarios:
    bf.add(u)

print(f'   Bloom Filter: {bf}')
print(f'   user_1    en filtro? {bf.contains("user_1")}     (esperado: True)')
print(f'   user_9999 en filtro? {bf.contains("user_9999")} (esperado: False)')
print(f'   Tasa FP real: {bf.expected_fp_rate()*100:.4f}% (objetivo: 1%)')
print(f'   Memoria: {bf.memory_bytes()/1024:.2f} KB vs ~{len(usuarios)*50//1024} KB dict aprox')

# Bot detector
bot = BotDetector(expected_users=10000)
bot.register_event('user_777', '192.168.1.1')
bot.register_event('user_777', '10.0.0.1')
bot.register_event('user_777', '172.16.0.1')
is_bot = bot.register_event('user_777', '8.8.8.8')
print(f'   user_777 con 4 IPs distintas → bot={is_bot} (esperado: True)')

In [ ]:
# ── TRIE ─────────────────────────────────────────────────────────────────────
print('='*55)
print('2. TRIE — Autocompletado de búsquedas')
print('='*55)

trie = Trie()
videos = [
    ('avengers endgame', 9500),
    ('avengers infinity war', 8900),
    ('avatar the way of water', 7200),
    ('avatar', 6100),
    ('the dark knight', 9800),
    ('the dark knight rises', 8100),
    ('thor ragnarok', 7800),
    ('spider-man no way home', 9100),
    ('spider-man homecoming', 7500),
]
for title, views in videos:
    trie.insert(title, frequency=views)

print('   Búsquedas por prefijo:')
for prefix in ['av', 'the d', 'spider', 'thor']:
    results = trie.autocomplete(prefix, top_k=3)
    print(f'     "{prefix}" → {results}')

In [ ]:
# ── COUNT-MIN SKETCH ─────────────────────────────────────────────────────────
print('='*55)
print('3. COUNT-MIN SKETCH — Top-K videos en tiempo real')
print('='*55)

import random
random.seed(42)

tracker = TopKTracker(k=10, epsilon=0.001, delta=0.01)
# Simular 100,000 reproducciones con distribución Zipf
catalogo = [f'video_{i:04d}' for i in range(1000)]
weights = [1/(i+1)**1.2 for i in range(len(catalogo))]
total_w = sum(weights)
weights = [w/total_w for w in weights]

for _ in range(100_000):
    video = random.choices(catalogo, weights=weights, k=1)[0]
    tracker.add_event(video)

print('   Top-10 videos más vistos (CMS estimado):')
for rank, (video, count) in enumerate(tracker.get_top_k(), 1):
    print(f'     #{rank:2d}  {video}  →  {count:,} reproducciones')

print(f'\n   Memoria CMS: {tracker.cms.memory_bytes()/1024:.1f} KB (constante sin importar cuántos videos)')

In [ ]:
# ── LRU CACHE ────────────────────────────────────────────────────────────────
print('='*55)
print('4. LRU CACHE — 1000 videos más accedidos en memoria')
print('='*55)

cache = LRUCache(capacity=5)  # Demo con cap=5

# Simular accesos
accesos = [
    ('v001', 'Avengers Endgame'),
    ('v002', 'Avatar'),
    ('v003', 'Dark Knight'),
    ('v004', 'Thor'),
    ('v005', 'Spider-Man'),
    ('v001', None),   # cache hit
    ('v006', 'Iron Man'),  # evict LRU
    ('v002', None),   # cache hit
]

print('   Simulando accesos con caché cap=5:')
for key, val in accesos:
    if val:
        cache.put(key, val)
        print(f'   PUT {key}: {val}')
    else:
        result = cache.get(key)
        status = 'HIT ✅' if result else 'MISS ❌'
        print(f'   GET {key}: {result} → {status}')

print(f'\n   Estado caché: {cache.get_lru_order()}')
print(f'   Hit rate: {cache.hit_rate*100:.1f}%')
print(f'   Stats: {cache.stats()}')

In [ ]:
# ── PRIORITY QUEUE ───────────────────────────────────────────────────────────
print('='*55)
print('5. PRIORITY QUEUE — Procesamiento por prioridades')
print('='*55)

from system.priority_queue import Priority

pq = PriorityQueue()

# Agregar eventos mixtos
eventos_test = [
    ('search',   'user_100', 'v001', False),  # estándar
    ('play',     'user_101', 'v002', True),   # premium
    ('purchase', 'user_102', 'v003', False),  # máxima prioridad
    ('play',     'user_103', 'v004', False),  # estándar
    ('play',     'user_104', 'v005', True),   # premium
    ('subscribe','user_105', None,   False),  # máxima prioridad
]

for etype, uid, vid, premium in eventos_test:
    e = Event.create(event_type=etype, user_id=uid, video_id=vid, is_premium=premium)
    pq.push(e)

print('   Orden de procesamiento (mayor prioridad primero):')
while pq:
    ev = pq.pop()
    prio_name = Priority(ev.priority).name
    print(f'   [{prio_name:10s}] {ev.event_type:12s} user={ev.user_id}')

In [ ]:
# ── MINHASH + LSH ────────────────────────────────────────────────────────────
print('='*55)
print('6. MINHASH + LSH — Recomendaciones por similitud')
print('='*55)

mh  = MinHash(num_perm=128)
lsh = LSH(num_perm=128, bands=32)

# Historial de videos vistos por cada usuario
usuarios_historial = {
    'Ana':    {'v001','v002','v003','v004','v005','v006'},
    'Bob':    {'v001','v002','v003','v007','v008','v009'},  # similar a Ana
    'Carlos': {'v010','v011','v012','v013','v014','v015'},  # diferente
    'Diana':  {'v001','v002','v004','v005','v016','v017'},  # algo similar a Ana
    'Eva':    {'v010','v011','v012','v018','v019','v020'},  # similar a Carlos
}

# Indexar usuarios
for nombre, historial in usuarios_historial.items():
    sig = mh.compute(historial)
    lsh.index(nombre, sig)

# Buscar usuarios similares a Ana
sig_ana = mh.compute(usuarios_historial['Ana'])
similares = lsh.query(sig_ana)
similares = [u for u in similares if u != 'Ana']

print('   Jaccard exacto vs MinHash estimado:')
for nombre, historial in usuarios_historial.items():
    if nombre == 'Ana': continue
    j_exact = MinHash.jaccard_exact(usuarios_historial['Ana'], historial)
    sig_b   = mh.compute(historial)
    j_estim = MinHash.jaccard_from_signatures(sig_ana, sig_b)
    flag = '<-- recomendado' if nombre in similares else ''
    print(f'   Ana <-> {nombre:8s}: exacto={j_exact:.3f}  estimado={j_estim:.3f}  {flag}')

print(f'\n   Usuarios recomendados para Ana: {similares}')

---
## Resumen Visual

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from system.priority_queue import Priority

sns.set_theme(style='darkgrid')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Sistema de Streaming de Video — Resumen de Estructuras', fontsize=14, fontweight='bold')

# 1. Bloom Filter: memoria vs elementos
ax = axes[0, 0]
ns = [1000, 10000, 100000, 1000000]
bf_mem_kb  = [BloomFilter(n=n, fp_rate=0.01).memory_bytes()/1024 for n in ns]
hs_mem_kb  = [n * 50 / 1024 for n in ns]
ax.loglog(ns, bf_mem_kb, 'o-', label='Bloom Filter', color='#2196F3', linewidth=2)
ax.loglog(ns, hs_mem_kb, 's-', label='Hash Set',     color='#FF5722', linewidth=2)
ax.set_title('Bloom Filter vs Hash Set — Memoria')
ax.set_xlabel('n (usuarios)')
ax.set_ylabel('Memoria (KB)')
ax.legend()

# 2. Priority Queue: distribución de eventos por prioridad
ax = axes[0, 1]
pq2 = PriorityQueue()
tipos = ['play']*60 + ['search']*25 + ['purchase']*5 + ['subscribe']*3 + ['pause']*7
prems = [random.random() < 0.2 for _ in tipos]
evs2  = [Event.create(t,'u',is_premium=p) for t,p in zip(tipos,prems)]
for e in evs2: pq2.push(e)
from collections import Counter
prio_counts = Counter()
while pq2:
    ev = pq2.pop()
    prio_counts[Priority(ev.priority).name] += 1
colors_pq = {'PAYMENT':'#E91E63','PREMIUM':'#9C27B0','STANDARD':'#2196F3','BACKGROUND':'#78909C'}
ax.bar(list(prio_counts.keys()), list(prio_counts.values()),
       color=[colors_pq.get(k,'gray') for k in prio_counts.keys()], alpha=0.85)
ax.set_title('Priority Queue — Distribución de Eventos')
ax.set_ylabel('Cantidad de eventos')

# 3. LRU Cache: hit rate por capacidad
ax = axes[1, 0]
caps = [10, 50, 100, 200, 500, 1000]
hit_rates_demo = []
stream_demo = random.choices(range(200), weights=[1/(i+1)**1.3 for i in range(200)], k=5000)
for cap in caps:
    c = LRUCache(capacity=cap)
    for vid in stream_demo:
        if c.get(str(vid)) is None:
            c.put(str(vid), vid)
    hit_rates_demo.append(c.hit_rate * 100)
ax.plot(caps, hit_rates_demo, 'o-', color='#4CAF50', linewidth=2, markersize=8)
ax.fill_between(caps, 0, hit_rates_demo, alpha=0.15, color='#4CAF50')
ax.axhline(y=80, linestyle='--', color='gray', alpha=0.6, label='80% objetivo')
ax.set_title('LRU Cache — Hit Rate vs Capacidad')
ax.set_xlabel('Capacidad del caché')
ax.set_ylabel('Hit rate (%)')
ax.set_ylim(0, 100)
ax.legend()

# 4. MinHash: error vs permutaciones
ax = axes[1, 1]
perm_vals = [16, 32, 64, 128, 256]
set_a = set(f'v{i}' for i in range(50))
set_b = set(f'v{i}' for i in range(30, 80))
j_real = MinHash.jaccard_exact(set_a, set_b)
errors = []
for np_val in perm_vals:
    mh_tmp = MinHash(num_perm=np_val)
    errs = [abs(MinHash.jaccard_from_signatures(mh_tmp.compute(set_a), mh_tmp.compute(set_b)) - j_real)
            for _ in range(50)]
    errors.append(np.mean(errs))
ax.plot(perm_vals, errors, 'o-', color='#FF9800', linewidth=2, markersize=8)
import numpy as np
ref = [errors[0] * np.sqrt(perm_vals[0]) / np.sqrt(p) for p in perm_vals]
ax.plot(perm_vals, ref, '--', color='gray', label='1/sqrt(k) teorico')
ax.axvline(x=128, color='green', linestyle=':', alpha=0.7, label='k=128 usado')
ax.set_title(f'MinHash — Error (J real={j_real:.2f})')
ax.set_xlabel('Numero de permutaciones (k)')
ax.set_ylabel('Error absoluto')
ax.legend()

plt.tight_layout()
os.makedirs('informe', exist_ok=True)
plt.savefig('informe/fig_resumen_demo.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura guardada en informe/fig_resumen_demo.png')

---
## Siguiente paso

Ahora que verificaste que todo funciona, abre el notebook de benchmarks completo:

```
notebooks/04_benchmarks.ipynb
```

**Nota:** El notebook de benchmarks usa la misma celda de setup (Celda 1 de este notebook).
Si los corres en la misma sesión de Colab, ya no necesitas re-instalar.